In [ ]:
import math, os, random, cv2, numpy, torch
import torch.nn as nn

1.Backbone

a) Conv

In [ ]:
class Conv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1,groups=1, activation=True):
      super().__init__()
      self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False, groups=groups)
      self.bn = nn.BatchNorm2d(out_channels, eps=0.001, momentum=0.03)
      self.act = nn.SiLU(inplace=True) if activation else nn.Identity()

    def forward(self,x):
      return self.act(self.bn(self.conv(x)))

b) C2f

In [ ]:
#Bottleneck: stack of 2 conv with shortcut connection (True/False):
class Bottleneck(nn.Module):
  def __init__(self, in_channels, out_channels, shortcut=True):
    super().__init__()
    self.conv1 = Conv(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
    self.conv2 = Conv(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
    self.shortcut = shortcut

  def forward(self, x):
    x_in = x #for residual connection
    x = self.conv1(x)
    x = self.conv2(x)
    if self.shortcut:
      x = x + x_in
    return x

#C2f: Conv + Bottleneck * N + Conv:
class C2f(nn.Module):
  def __init__(self, in_channels, out_channels, num_bottlenecks, shortcut=True):
    super().__init__()
    self.mid_channels = out_channels // 2
    self.num_bottlenecks = num_bottlenecks
    self.conv1 = Conv(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    #sequence of bottleneck layers
    self.m = nn.ModuleList([Bottleneck(self.mid_channels, self.mid_channels) for _ in range(self.num_bottlenecks)])

    self.conv2 = Conv((num_bottlenecks+2)*self.mid_channels, out_channels, kernel_size=1, stride=1, padding=0)

  def forward(self, x):
    x = self.conv1(x)

    #split x along channel dimension
    x1, x2 = x[:, :x.shape[1]//2, :, :], x[:, x.shape[1]//2:, :, :]

    #list of outputs
    outputs=[x1, x2] #x1 is fed through the bottlenecks

    for i in range(self.num_bottlenecks):
      x1 = self.m[i](x1)
      outputs.insert(0, x1)

    outputs = torch.cat(outputs, dim=1)
    out = self.conv2(outputs)

    return out

c) SPPF

In [ ]:
class SPPF(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=5):
    #kernel_size = size of maxpool
    super().__init__()
    hidden_channels = in_channels // 2
    self.conv1 = Conv(in_channels, hidden_channels, kernel_size=1, stride=1, padding=0)
    #concatenate outputs of maxpool and feed to conv2
    self.conv2 = Conv(4*hidden_channels, out_channels, kernel_size=1, stride=1, padding=0)
    #maxpool is applied at 3 different scales
    self.m = nn.MaxPool2d(kernel_size=kernel_size, stride=1, padding=kernel_size//2, dilation=1, ceil_mode=False)

  def forward(self, x):
    x = self.conv1(x)
    x1 = self.m(x)
    x2 = self.m(x1)
    x3 = self.m(x2)
    x = torch.cat([x, x1, x2, x3], dim=1)
    x = self.conv2(x)
    return x

Putting things together

In [ ]:
#backbone = DarkNet53

#return d, w, r
def yolo_params(version):
  if version == 'n':
    return 1/3, 1/4, 2.0
  elif version == 's':
    return 1/3, 1/2, 2.0
  elif version == 'm':
    return 2/3, 3/4, 1.5
  elif version == 'l':
    return 1.0, 1.0, 1.0
  elif version == 'x':
    return 1.0, 1.25, 1.0

class Backbone(nn.Module):
  def __init__(self, version, in_channels=3, shortcut=True):
     super().__init__()
     d, w, r = yolo_params(version)

     #conv layers
     self.conv0 = Conv(in_channels, int(64*w), kernel_size=3, stride=2, padding=1)
     self.conv1 = Conv(int(64*w), int(128*w), kernel_size=3, stride=2, padding=1)
     self.conv3 = Conv(int(128*w), int(256*w), kernel_size=3, stride=2, padding=1)
     self.conv5 = Conv(int(256*w), int(512*w), kernel_size=3, stride=2, padding=1)
     self.conv7 = Conv(int(512*w), int(512*w*r), kernel_size=3, stride=2, padding=1)


     #c2f layers
     self.c2f2 = C2f(int(128*w), int(128*w), int(3*d), shortcut)
     self.c2f4 = C2f(int(256*w), int(256*w), int(6*d), shortcut)
     self.c2f6 = C2f(int(512*w), int(512*w), int(6*d), shortcut)
     self.c2f8 = C2f(int(512*w*r), int(512*w*r), int(3*d), shortcut)

     #sppf layer
     self.sppf = SPPF(int(512*w*r), int(512*w*r))

  def forward(self, x):
    x = self.conv0(x)
    x = self.conv1(x)
    x = self.c2f2(x)
    x = self.conv3(x)
    out1 = self.c2f4(x) #keep for output
    x = self.conv5(out1)
    out2 = self.c2f6(x) #keep for output
    x = self.conv7(out2)
    x = self.c2f8(x)
    out3 = self.sppf(x)

    return out1, out2, out3

print("----Nano model----")
backbone_n = Backbone('n')
print(f"{sum(p.numel() for p in backbone_n.parameters())/1e6} million parameters")

print("----Small model----")
backbone_s = Backbone('s')
print(f"{sum(p.numel() for p in backbone_s.parameters())/1e6} million parameters")

----Nano model----
1.272656 million parameters
----Small model----
5.079712 million parameters


2. Neck

The neck comprises of Upsample + C2f

Upsample

In [ ]:
# upsample = nearest-neighbor interpolation with scale_factor=2. It doesn't have trainable parameters.
class Upsample(nn.Module):
  def __init__(self, scale_factor=2, mode='nearest'):
    super().__init__()
    self.scale_factor = scale_factor
    self.mode = mode
  def forward(self, x):
    return nn.functional.interpolate(x, scale_factor=self.scale_factor, mode=self.mode)

In [ ]:
class Neck(nn.Module):
  def __init__(self, version):
    super().__init__()
    d, w, r = yolo_params(version)
    self.upsample = Upsample()

    self.c2f1 = C2f(int(512*w*(1+r)), int(512*w), int(3*d), shortcut=False)
    self.c2f2 = C2f(int(768*w), int(256*w), int(3*d), shortcut=False)
    self.c2f3 = C2f(int(768*w), int(512*w), int(3*d), shortcut=False)
    self.c2f4 = C2f(int(512*w*(1+r)), int(512*w*r), int(3*d), shortcut=False)

    self.conv1 = Conv(int(256*w), int(256*w), kernel_size=3, stride=2, padding=1)
    self.conv2 = Conv(int(512*w), int(512*w), kernel_size=3, stride=2, padding=1)

  def forward(self, x_res1, x_res2, x):
    #x_rest1, x_rest2 = outputs of backbone
    res1 = x #for residual connection

    x = self.upsample(x)
    x = torch.cat([x, x_res2], dim=1)

    res2 = self.c2f1(x) #for residual connection

    x = self.upsample(res2)
    x = torch.cat([x, x_res1], dim=1)
    out1 = self.c2f2(x)

    x = self.conv1(out1)
    x = torch.cat([x, res2], dim=1)
    out2 = self.c2f3(x)

    x = self.conv2(out2)
    x = torch.cat([x, res1], dim=1)
    out3 = self.c2f4(x)

    return out1, out2, out3

neck=Neck(version='n')
print(f"{sum(p.numel() for p in neck.parameters())/1e6} million parameters")
x=torch.rand((1,3,640,640))
out1,out2,out3=Backbone(version='n')(x)
out_1,out_2,out_3=neck(out1,out2,out3)
print(out_1.shape)
print(out_2.shape)
print(out_3.shape)

0.98688 million parameters
torch.Size([1, 64, 80, 80])
torch.Size([1, 128, 40, 40])
torch.Size([1, 256, 20, 20])


3. Head

Distribution Focal Loss (DFL) represents each bounding box coordinate as a discrete probability distribution instead of predicting a single continuous value directly. For each side of the box, YOLOv8 predicts probabilities over 16 bins and then computes the final coordinate as the expected value of that distribution. This makes bounding box regression smoother and more precise, especially when the object boundary lies between two discrete positions.


a) DFL

In [ ]:
#DFL
class DFL(nn.Module):
  def __init__(self, ch=16):
      super().__init__()
      self.ch = ch
      self.conv = nn.Conv2d(in_channels=ch, out_channels=1, kernel_size=1, bias=False).requires_grad_(False) #don't learn parameters

      #initialize conv with [0, ..., ch-1]
      x = torch.arange(ch, dtype=torch.float).view(1, ch, 1, 1)
      self.conv.weight.data[:] = torch.nn.Parameter(x) #DFL only has ch parameters

  def forward(self, x):
      #x must have num_channels = 4*ch: x = [bs, 4*ch, c]
      b, c, a = x.shape #c = 4*ch
      x = x.view(b, 4, self.ch, a).transpose(1, 2) #x = [bs, ch, 4, c]
      x = x.softmax(1)
      x = self.conv(x)
      return x.view(b, 4, a)

# sanity check
dummy_input=torch.rand((1,64,128))
dfl=DFL()
print(f"{sum(p.numel() for p in dfl.parameters())} parameters")

dummy_output=dfl(dummy_input)
print(dummy_output.shape)

print(dfl)


16 parameters
torch.Size([1, 4, 128])
DFL(
  (conv): Conv2d(16, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
)


b) Head

In [ ]:
class Head(nn.Module):
    def __init__(self,version,ch=16,num_classes=80):

        super().__init__()
        self.ch=ch                          # dfl channels
        self.coordinates=self.ch*4          # number of bounding box coordinates
        self.nc=num_classes                 # 80 for COCO
        self.no=self.coordinates+self.nc    # number of outputs per prediction point

        self.stride=torch.zeros(3)          # strides computed during build

        d,w,r=yolo_params(version=version)

        # for bounding boxes
        self.box=nn.ModuleList([
            nn.Sequential(Conv(int(256*w),self.coordinates,kernel_size=3,stride=1,padding=1),
                          Conv(self.coordinates,self.coordinates,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.coordinates,self.coordinates,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w),self.coordinates,kernel_size=3,stride=1,padding=1),
                          Conv(self.coordinates,self.coordinates,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.coordinates,self.coordinates,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w*r),self.coordinates,kernel_size=3,stride=1,padding=1),
                          Conv(self.coordinates,self.coordinates,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.coordinates,self.coordinates,kernel_size=1,stride=1))
        ])

        # for classification
        self.cls=nn.ModuleList([
            nn.Sequential(Conv(int(256*w),self.nc,kernel_size=3,stride=1,padding=1),
                          Conv(self.nc,self.nc,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.nc,self.nc,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w),self.nc,kernel_size=3,stride=1,padding=1),
                          Conv(self.nc,self.nc,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.nc,self.nc,kernel_size=1,stride=1)),

            nn.Sequential(Conv(int(512*w*r),self.nc,kernel_size=3,stride=1,padding=1),
                          Conv(self.nc,self.nc,kernel_size=3,stride=1,padding=1),
                          nn.Conv2d(self.nc,self.nc,kernel_size=1,stride=1))
        ])

        # dfl
        self.dfl = DFL(self.ch)

    def forward(self,x):
        # x = output of Neck = list of 3 tensors with different resolution and different channel dim
        #     x[0]=[bs, ch0, w0, h0], x[1]=[bs, ch1, w1, h1], x[2]=[bs,ch2, w2, h2]

        for i in range(len(self.box)):       # detection head i
            box=self.box[i](x[i])            # [bs,num_coordinates,w,h]
            cls=self.cls[i](x[i])            # [bs,num_classes,w,h]
            x[i]=torch.cat((box,cls),dim=1)  # [bs,num_coordinates+num_classes,w,h]

        # in training, no dfl output
        if self.training:
            return x                         # [3,bs,num_coordinates+num_classes,w,h]

        # in inference time, dfl produces refined bounding box coordinates
        anchors, strides = (i.transpose(0, 1) for i in self.make_anchors(x, self.stride))

        # concatenate predictions from all detection layers
        x = torch.cat([i.view(x[0].shape[0], self.no, -1) for i in x], dim=2) #[bs, 4*self.ch + self.nc, sum_i(h[i]w[i])]

        # split out predictions for box and cls
        #           box=[bs,4×self.ch,sum_i(h[i]w[i])]
        #           cls=[bs,self.nc,sum_i(h[i]w[i])]
        box, cls = x.split(split_size=(4 * self.ch, self.nc), dim=1)


        a, b = self.dfl(box).chunk(2, 1)  # a,b = [bs, 2, total_points]
        a = anchors.unsqueeze(0) - a
        b = anchors.unsqueeze(0) + b
        box = torch.cat(tensors=((a + b) / 2, b - a), dim=1)

        return torch.cat(tensors=(box * strides, cls.sigmoid()), dim=1)


    def make_anchors(self, x, strides, offset=0.5):
        # x= list of feature maps: x=[x[0],...,x[N-1]], in our case N= num_detection_heads=3
        #                          each having shape [bs,ch,w,h]
        #    each feature map x[i] gives output[i] = w*h anchor coordinates + w*h stride values

        # strides = list of stride values indicating how much
        #           the spatial resolution of the feature map is reduced compared to the original image

        assert x is not None
        anchor_tensor, stride_tensor = [], []
        dtype, device = x[0].dtype, x[0].device
        for i, stride in enumerate(strides):
            _, _, h, w = x[i].shape
            sx = torch.arange(end=w, device=device, dtype=dtype) + offset  # x coordinates of anchor centers
            sy = torch.arange(end=h, device=device, dtype=dtype) + offset  # y coordinates of anchor centers
            sy, sx = torch.meshgrid(sy, sx)                                # all anchor centers
            anchor_tensor.append(torch.stack((sx, sy), -1).view(-1, 2))
            stride_tensor.append(torch.full((h * w, 1), stride, dtype=dtype, device=device))
        return torch.cat(anchor_tensor), torch.cat(stride_tensor)

4. Putting everything together

In [ ]:
class MyYolo(nn.Module):
  def __init__(self, version):
    super().__init__()
    self.backbone = Backbone(version=version)
    self.neck = Neck(version=version)
    self.head = Head(version=version)

  def forward(self, x):
    x = self.backbone(x)            #return out1, out2, out3
    x = self.neck(x[0], x[1], x[2]) #return out_1, out_2, out_3
    return self.head(list(x))

model = MyYolo(version='n')
print(f"{sum(p.numel() for p in model.parameters())/1e6} million parameters")
print(model)

10.497824 million parameters
MyYolo(
  (backbone): Backbone(
    (conv0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (conv1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (conv3): Conv(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (conv5): Conv(
      (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
     